In [9]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark import StorageLevel
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from itertools import product

import time
import uuid

In [10]:
# Use the existing Dataproc cluster.
spark = (
    SparkSession.builder
    .appName("Rainfall_GBT_Experiments")
    .getOrCreate()
)
 
spark.conf.set("spark.sql.session.timeZone", "Asia/Singapore")
spark.conf.set("spark.sql.shuffle.partitions", "128")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

ML_PARTITIONS = 32
 
print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
print("Training partitions:", ML_PARTITIONS) 

Spark version: 3.5.3
Spark master: yarn
Shuffle partitions: 128
Training partitions: 32


In [11]:
train_final = spark.read.parquet("gs://dsa5208-rainfall-sixing-data/train_parquet/")
val_final = spark.read.parquet("gs://dsa5208-rainfall-sixing-data/val_parquet/")
test_final = spark.read.parquet("gs://dsa5208-rainfall-sixing-data/test_parquet/")

In [12]:
import uuid

# Create a unique test location in Sixing's bucket.
check_path = (
    "gs://dsa5208-rainfall-sixing-data/task2-ashley/_write_check_"
    + uuid.uuid4().hex
)

# Save one row to check access before retraining.
spark.range(1).write.mode("overwrite").parquet(check_path)

print("Write access works!")

Write access works!


In [13]:
feature_cols = [
    "location_latitude",
    "location_longitude",
    "hour",
    "month",
    "reading_value",
    "lag_5m",
    "lag_30m",
    "lag_1h",
    "rain_cumsum_30m",
    "rain_cumsum_1h",
    "wet_count_30m",
    "thr"
]

THRESHOLDS = [0.0, 1.0, 2.5, 5.0, 10.0]

In [14]:
# Create a separate output folder for this experiment.
'''
OUTPUT = (
    "gs://dsa5208-rainfall-sixing-data/task2/gbt_runs/"
    + uuid.uuid4().hex[:8]
)
 
RESULTS_PATH = OUTPUT + "/results"
print("Saving to:", OUTPUT)
'''
# These metrics evaluate probability rankings; no F1 is calculated.
pr_evaluator = BinaryClassificationEvaluator(metricName="areaUnderPR")
roc_evaluator = BinaryClassificationEvaluator(metricName="areaUnderROC")
 
 
def train_gbt(depth, learning_rate, iterations):
 
    # STEP 1: Combine the input columns into a feature vector.
    assembler = VectorAssembler(
        inputCols=feature_cols,
        outputCol="features"
    )
 
    # STEP 2: Configure this grid combination.
    gbt = GBTClassifier(
        maxDepth=depth,
        stepSize=learning_rate,
        maxIter=iterations,
        seed=42
    )
 
    pipeline = Pipeline(stages=[assembler, gbt])
 
    # STEP 3: Train using the prepared training sample.
    # This timer measures fitting only, excluding evaluation and saving.
    start = time.perf_counter()
    model = pipeline.fit(train_final)
    training_seconds = time.perf_counter() - start
 
    # STEP 4: Generate validation predictions.
    # Cache them because the metric calculations reuse them.
    predictions = (
        model.transform(val_final)
        .select("thr", "label", "rawPrediction")
        .persist(StorageLevel.MEMORY_AND_DISK)
    )
 
    pr_evaluator = BinaryClassificationEvaluator(
        metricName="areaUnderPR"
    )
 
    roc_evaluator = BinaryClassificationEvaluator(
        metricName="areaUnderROC"
    )
 
    rows = []
 
    try:
        # STEP 5: Evaluate each rainfall threshold separately.
        for threshold in THRESHOLDS:
 
            subset = predictions.filter(
                F.col("thr") == threshold
            )
 
            auc_pr = pr_evaluator.evaluate(subset)
            auc_roc = roc_evaluator.evaluate(subset)
 
            # Record every tuned parameter in its own column.
            rows.append({
                "maxDepth": int(depth),
                "stepSize": float(learning_rate),
                "maxIter": int(iterations),
                "seed": 42,
                "thr": float(threshold),
                "auc_pr": float(auc_pr),
                "auc_roc": float(auc_roc),
                "training_seconds": float(training_seconds)
            })
 
        # STEP 6: Append this model's five result rows.
        # Existing results in RESULTS_PATH are retained.
        run_results = spark.createDataFrame(rows)
 
        (
            run_results
            .coalesce(1)
            .write
            .mode("append")
            .parquet(RESULTS_PATH)
        )
 
    finally:
        # Release this model's prediction cache, even if evaluation fails.
        predictions.unpersist()
 
    # STEP 7: Calculate the score used to compare grid combinations.
    # Each rainfall threshold contributes equally.
    average_pr = sum(row["auc_pr"] for row in rows) / len(rows)
 
    print(
        f"Depth={depth}, "
        f"Learning rate={learning_rate}, "
        f"Iterations={iterations} | "
        f"Average PR-AUC={average_pr:.4f} | "
        f"Training time={training_seconds:.1f}s"
    )
 
    # The grid uses these to keep the best model.
    return model, average_pr

In [15]:
# STEP 1: Define all combis
DEPTHS = [4, 5, 6]
LEARNING_RATES = [0.05, 0.1]
ITERATIONS = [60, 80, 100, 120, 150]

# 3 × 2 × 5 = 30 models
grid = list(product(DEPTHS, LEARNING_RATES, ITERATIONS))

print("Number of models:", len(grid))
 
print("Number of models:", len(grid))
 
 
# STEP 2: Create a fresh results folder.
# Your existing train_gbt() function uses RESULTS_PATH when saving results.
OUTPUT = (
    "gs://dsa5208-rainfall-sixing-data/task2-ashley/gbt_runs/"
    + uuid.uuid4().hex
)
RESULTS_PATH = OUTPUT + "/results_by_threshold"
 
print("Save location:", OUTPUT)
 
 
# STEP 3: Start without a selected model.
best_model = None
best_pr = float("-inf")
best_parameters = None
 
 
# STEP 4: Train every combination sequentially.
for run, (depth, rate, iterations) in enumerate(grid, start=1):
 
    print(
        f"\nRun {run}/{len(grid)}: "
        f"maxDepth={depth}, stepSize={rate}, maxIter={iterations}"
    )
 
    # Fits the model, evaluates validation data and saves its results.
    model, average_pr = train_gbt(
        depth=depth,
        learning_rate=rate,
        iterations=iterations
    )
 
    # Keep the highest average validation PR-AUC.
    if average_pr > best_pr:
        best_model = model
        best_pr = average_pr
 
        best_parameters = {
            "maxDepth": depth,
            "stepSize": rate,
            "maxIter": iterations
        }
 
        # Save whenever a better model is found.
        best_model.write().overwrite().save(OUTPUT + "/best_model")
 
    print("Best parameters so far:", best_parameters)
    print("Best average validation PR-AUC:", round(best_pr, 4))
 
 
# STEP 5: Summarise the saved results.
results = spark.read.parquet(RESULTS_PATH)
 
# One row per combination.
# AUC scores are averaged across the five rainfall thresholds.
summary = (
    results
    .groupBy("maxDepth", "stepSize", "maxIter", "seed")
    .agg(
        F.avg("auc_pr").alias("auc_pr"),
        F.avg("auc_roc").alias("auc_roc"),
        F.first("training_seconds").alias("training_seconds")
    )
)
 
summary.orderBy(F.desc("auc_pr")).show(20, truncate=False)
 
 
# STEP 6: Save the summary.
summary.coalesce(1).write.mode("overwrite").parquet(
    OUTPUT + "/summary"
)
 
print("Final selected parameters:", best_parameters)
print("Final average validation PR-AUC:", best_pr)
print("Results saved under:", OUTPUT)

Number of models: 30
Number of models: 30
Save location: gs://dsa5208-rainfall-sixing-data/task2-ashley/gbt_runs/696ac8b3c6634ed8802f536e96236901

Run 1/30: maxDepth=4, stepSize=0.05, maxIter=60


26/10/15 15:37:16 WARN SimpleTableFunctionRegistry: The function read_files replaced a previously registered function.


Depth=4, Learning rate=0.05, Iterations=60 | Average PR-AUC=0.4740 | Training time=67.8s


Best parameters so far: {'maxDepth': 4, 'stepSize': 0.05, 'maxIter': 60}
Best average validation PR-AUC: 0.474

Run 2/30: maxDepth=4, stepSize=0.05, maxIter=80


26/10/15 15:39:01 WARN SimpleTableFunctionRegistry: The function read_files replaced a previously registered function.


...

26/10/15 16:09:16 WARN SimpleTableFunctionRegistry: The function read_files replaced a previously registered function.


Depth=5, Learning rate=0.1, Iterations=80 | Average PR-AUC=0.4686 | Training time=90.7s
Best parameters so far: {'maxDepth': 4, 'stepSize': 0.05, 'maxIter': 60}
Best average validation PR-AUC: 0.474

Run 18/30: maxDepth=5, stepSize=0.1, maxIter=100


ERROR:root:KeyboardInterrupt while sending command.                             
Traceback (most recent call last):
  File "/usr/lib/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/miniconda3/lib/python3.11/socket.py", line 718, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

26/10/15 16:11:25 WARN TransportChannelHandler: Exception in connection from /10.148.0.46:35098
java.io.IOException: Connection reset by peer
	at java.base/sun.nio.ch.FileDispatcherImpl.read0(Native Method) ~[?:?]
	at java.base/sun.nio.ch.SocketDispatcher.read(SocketDispatcher.java:39) ~[?:?]
	at java.base/sun.nio.ch.IOUtil.readIntoNativeBuffer(IOUtil.java:276) ~[?:?]
	at java.base/sun.nio.ch.IOUtil.read(IOUtil.java:233) ~[?:?]
	at java.base/sun.nio.ch.IOUtil.read(IOUtil.java:223) ~[?:?]
	at java.base/sun.nio.ch.SocketChannelImpl.read(SocketChannelImpl.java:356) ~[?:?]
	at io.netty.buffer.PooledByteBuf.setBytes(PooledByteBuf.java:254) ~[netty-buffer-4.1.100.Final.jar:4.1.100.Final]
	at io.netty.buffer.AbstractByteBuf.writeBytes(AbstractByteBuf.java:1132) ~[netty-buffer-4.1.100.Final.jar:4.1.100.Final]
	at io.netty.channel.socket.nio.NioSocketChannel.doReadBytes(NioSocketChannel.java:357) ~[netty-transport-4.1.100.Final.jar:4.1.100.Final]
	at io.netty.channel.nio.AbstractNioByteChannel$